# Askvocate — Smart Lawyer Recommendation System (Approach A)

## 📌 System Architecture Overview
1. **User Prompt**: The user types their legal situation in plain language.
2. **Automatic Location Injection**: `user_city` & `user_state` are automatically retrieved from User DB context — **no location typed by user**.
3. **Domain Classification**: The system detects the legal domain (e.g. *Child Protection Law*, *Motor Accident Claims*, *Medical Negligence Law*).
4. **Tiered Primary + Secondary Match & Affordability Ranking**: Recommends lawyers whose `practice_area_primary` **or** `practice_area_secondary` matches the detected domain, with primary specialists weighted higher, ordered so **best & most affordable lawyers appear first**.


In [17]:
import pandas as pd
import numpy as np
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', 15)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 80)
print("Libraries loaded successfully!")

Libraries loaded successfully!


## 1. Load Datasets
Load the clean case dataset (`cases_2_fixed.csv`), advocate directory (`advocate_details_fixed.csv`), and test queries.

In [18]:
cases = pd.read_csv('cases_2_fixed.csv', encoding='utf-8')
adv   = pd.read_csv('advocate_details_fixed.csv', encoding='utf-8')
test_queries = pd.read_csv('test_queries_fixed.csv', encoding='utf-8')

print(f"Loaded Cases Dataset     : {cases.shape[0]} rows | {cases['legal_domain'].nunique()} unique unclubbed domains")
print(f"Loaded Advocates Directory: {adv.shape[0]} lawyers | {adv['practice_area_primary'].nunique()} unique primary practice domains")
print(f"Loaded Test Queries      : {test_queries.shape[0]} sample queries")

Loaded Cases Dataset     : 11500 rows | 36 unique unclubbed domains
Loaded Advocates Directory: 7000 lawyers | 37 unique primary practice domains
Loaded Test Queries      : 666 sample queries


## 2. Build Domain Knowledge Profiles
Construct text knowledge profiles for each of the 36 unclubbed legal domains directly from case law facts, issues, and keywords.

In [19]:
cases['combo_text'] = (
    cases['legal_issue'].fillna('') + ' ' +
    cases['case_facts_summary'].fillna('') + ' ' +
    cases['search_keywords'].fillna('')
)

FULL_PROFILES = cases.groupby('legal_domain')['combo_text'].apply(lambda x: ' '.join(x)).to_dict()
categories = list(FULL_PROFILES.keys())

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
domain_vectors = vectorizer.fit_transform([FULL_PROFILES[c] for c in categories])

print(f"Domain Knowledge Base built across {len(categories)} unclubbed legal domains!")

Domain Knowledge Base built across 36 unclubbed legal domains!


## 3. Approach A: Domain Classifier
Predicts the target legal domain from the user's prompt text.

In [20]:
def predict_legal_domains(prompt_text, top_k=1, min_confidence=0.01):
    """
    Predicts Top-K legal domains for a given user prompt text.
    """
    vec = vectorizer.transform([prompt_text])
    sims = cosine_similarity(vec, domain_vectors)[0]
    top_indices = sims.argsort()[::-1][:top_k]
    
    predictions = []
    for idx in top_indices:
        score = round(float(sims[idx]), 4)
        if score >= min_confidence:
            predictions.append(categories[idx])
            
    if not predictions:
        predictions = [categories[top_indices[0]]]
        
    return predictions

## 4. Tiered Primary + Secondary Match & Affordability Ranking Engine (Displays All Lawyers)
1. Matches lawyers whose **`practice_area_primary` matches the detected domain (full specialization score)**, or whose **`practice_area_secondary` list contains the detected domain (weighted specialization score)**.
2. Auto-injects `user_city` / `user_state` from User DB.
3. Ranks and displays **ALL matching lawyers**, ordered so **best & most affordable lawyers appear first**.

**Why add secondary matching?** Many advocates list 2–3 adjacent practice areas in `practice_area_secondary`. Restricting to primary-only means that whenever a city/state has zero primary specialists in a domain, the old fallback jumped straight to showing lawyers with *no relevant specialization at all*. Secondary matching adds a much more relevant middle tier before that last-resort fallback, and is weighted lower than a true primary-specialist match so specialists still rank first.


In [21]:
def recommend_lawyers_approach_a(user_prompt, user_city, user_state=None, top_k=None,
                                  secondary_weight=0.65):
    """
    Approach A Recommendation Engine (Primary + Secondary Practice Area Matching):
    - Matches advocate's practice_area_primary (full specialization score = 1.0)
    - Also matches practice_area_secondary as a weighted tier (specialization score = secondary_weight)
    - Auto-injects user_city from DB context
    - Affordability-First Ranking (best & cheapest lawyers displayed first)
    - Displays ALL matching lawyers (or top_k if specified)
    """
    predicted_domains = predict_legal_domains(user_prompt, top_k=1)
    target_domain = predicted_domains[0]

    candidates = adv.copy()

    # 1. Location Filter
    city_pool = candidates[candidates['city'].str.lower() == str(user_city).lower()].copy()
    if len(city_pool) == 0 and user_state:
        city_pool = candidates[candidates['state'].str.lower() == str(user_state).lower()].copy()
    if len(city_pool) == 0:
        city_pool = candidates.copy()

    city_pool = city_pool[city_pool['availability'] != 'Not accepting new matters'].copy()

    def _secondary_match(sec_val):
        if pd.isna(sec_val):
            return False
        parts = [p.strip() for p in str(sec_val).split(';')]
        return target_domain in parts

    def _match_tier(pool):
        """Split a candidate pool into weighted primary + secondary matches."""
        primary_hits = pool[pool['practice_area_primary'] == target_domain].copy()
        primary_hits['score_specialization'] = 1.0
        primary_hits['match_tier'] = 'primary'

        remaining = pool.drop(primary_hits.index)
        secondary_mask = remaining['practice_area_secondary'].apply(_secondary_match)
        secondary_hits = remaining[secondary_mask].copy()
        secondary_hits['score_specialization'] = secondary_weight
        secondary_hits['match_tier'] = 'secondary'

        return pd.concat([primary_hits, secondary_hits], ignore_index=False)

    # 2. PRIMARY + SECONDARY Practice Area Match (city level)
    matched = _match_tier(city_pool)

    # 3. Fallback to state level if no primary/secondary match in exact city
    if len(matched) == 0 and user_state:
        state_pool = candidates[candidates['state'].str.lower() == str(user_state).lower()].copy()
        state_pool = state_pool[state_pool['availability'] != 'Not accepting new matters'].copy()
        matched = _match_tier(state_pool)

    # 4. Last-resort fallback: no domain match anywhere in city/state -> show unfiltered pool,
    #    clearly flagged so the caller/UI can warn the user these aren't domain-matched.
    if len(matched) == 0:
        matched = city_pool.copy()
        matched['score_specialization'] = 0.0
        matched['match_tier'] = 'unmatched'

    city_candidates = matched

    # 5. Affordability Score (Cheaper consultation fee = Higher score)
    min_fee = city_candidates['consultation_fee_inr'].min()
    max_fee = city_candidates['consultation_fee_inr'].max()
    if max_fee > min_fee:
        city_candidates['score_affordability'] = 1.0 - ((city_candidates['consultation_fee_inr'] - min_fee) / (max_fee - min_fee))
    else:
        city_candidates['score_affordability'] = 1.0

    # 6. Rating & Experience Scores
    city_candidates['score_rating'] = city_candidates['profile_rating'].fillna(3.5) / 5.0
    city_candidates['score_experience'] = city_candidates['years_of_experience'].clip(upper=25) / 25.0

    # 7. Composite Ranking Score
    city_candidates['ranking_score'] = (
        0.40 * city_candidates['score_specialization'] +
        0.25 * city_candidates['score_affordability'] +
        0.20 * city_candidates['score_rating'] +
        0.15 * city_candidates['score_experience']
    )

    # Sort by ranking score descending (Best & Most Affordable First)
    result = city_candidates.sort_values(['ranking_score', 'consultation_fee_inr'], ascending=[False, True])

    res_cols = [
        'advocate_name', 'practice_area_primary', 'practice_area_secondary', 'match_tier',
        'consultation_fee_inr', 'profile_rating', 'years_of_experience',
        'city', 'availability', 'ranking_score'
    ]

    if top_k is not None:
        return predicted_domains, result[res_cols].head(top_k)
    return predicted_domains, result[res_cols]


## 5. End-to-End Practical Demonstrations
Live demonstrations showing how raw user prompts + DB location context display **ALL matching primary specialists, ordered by affordability**.

In [22]:
# --- DEMO 1: Child Protection Case ---
user_db_context_1 = {"user_id": "USR-101", "city": "Delhi", "state": "Delhi"}
user_prompt_1 = "My neighbor harassed my child"

domains_1, lawyers_1 = recommend_lawyers_approach_a(
    user_prompt=user_prompt_1,
    user_city=user_db_context_1["city"],
    user_state=user_db_context_1["state"]
)

print("===========================================================")
print(f"User Prompt      : '{user_prompt_1}'")
print(f"User City (DB)   : {user_db_context_1['city']}")
print(f"Detected Domain  : {domains_1}")
print(f"Total Matching Lawyers Found: {len(lawyers_1)}  "
      f"(primary: {(lawyers_1['match_tier']=='primary').sum()}, "
      f"secondary: {(lawyers_1['match_tier']=='secondary').sum()})")
print("===========================================================")
print("ALL Matching Specialists (Primary + Secondary, Best & Most Affordable First):")
display(lawyers_1)

# --- DEMO 2: Motor Accident Claim ---
user_db_context_2 = {"user_id": "USR-102", "city": "Mumbai", "state": "Maharashtra"}
user_prompt_2 = "Car accident happened yesterday, looking for claim compensation lawyer"

domains_2, lawyers_2 = recommend_lawyers_approach_a(
    user_prompt=user_prompt_2,
    user_city=user_db_context_2["city"],
    user_state=user_db_context_2["state"]
)

print("\n===========================================================")
print(f"User Prompt      : '{user_prompt_2}'")
print(f"User City (DB)   : {user_db_context_2['city']}")
print(f"Detected Domain  : {domains_2}")
print(f"Total Matching Lawyers Found: {len(lawyers_2)}  "
      f"(primary: {(lawyers_2['match_tier']=='primary').sum()}, "
      f"secondary: {(lawyers_2['match_tier']=='secondary').sum()})")
print("===========================================================")
print("ALL Matching Specialists (Primary + Secondary, Best & Most Affordable First):")
display(lawyers_2)

# --- DEMO 3: Secondary-Match Benefit Case ---
# New Delhi has ZERO advocates whose practice_area_primary is "Motor Accident Claims",
# so the old primary-only engine would have skipped straight to an unfiltered,
# domain-irrelevant fallback here. With secondary matching, relevant lawyers are found.
user_db_context_3 = {"user_id": "USR-103", "city": "", "state": ""}
user_prompt_3 = "Truck hit my bike on the highway, I need help claiming compensation"

domains_3, lawyers_3 = recommend_lawyers_approach_a(
    user_prompt=user_prompt_3,
    user_city=user_db_context_3["city"],
    user_state=user_db_context_3["state"]
)

print("\n===========================================================")
print(f"User Prompt      : '{user_prompt_3}'")
print(f"User City (DB)   : {user_db_context_3['city']}")
print(f"Detected Domain  : {domains_3}")
print(f"Total Matching Lawyers Found: {len(lawyers_3)}  "
      f"(primary: {(lawyers_3['match_tier']=='primary').sum()}, "
      f"secondary: {(lawyers_3['match_tier']=='secondary').sum()})")
print("Note: 0 primary specialists exist in this city for this domain -- every result")
print("below was only surfaced because of the new secondary-match tier.")
print("===========================================================")
display(lawyers_3)


User Prompt      : 'My neighbor harassed my child'
User City (DB)   : Delhi
Detected Domain  : ['Child Protection Law']
Total Matching Lawyers Found: 8  (primary: 1, secondary: 7)
ALL Matching Specialists (Primary + Secondary, Best & Most Affordable First):


,advocate_name,practice_area_primary,practice_area_secondary,match_tier,consultation_fee_inr,profile_rating,years_of_experience,city,availability,ranking_score
1814,Adv. Simran Naik,Corporate & Commercial Law,Criminal Law; Information Technology Law; Child Protection Law,secondary,1000,3.8,28,New Delhi,Limited availability,0.784222
4792,Adv. Nidhi Reddy,Real Estate & Housing Law,Criminal Law; Civil Litigation; Child Protection Law,secondary,1000,3.6,31,New Delhi,Available for new matters,0.776222
1090,Adv. Neha Mukherjee,White Collar Crime,Constitutional Law; Consumer Law; Child Protection Law,secondary,500,4.7,12,New Delhi,Available for new matters,0.770000
624,Adv. Nidhi Sharma,GST & Tax Law,Criminal Law; Data Protection and Privacy; Child Protection Law,secondary,2500,4.9,37,New Delhi,Available for new matters,0.744889
1875,Adv. Ishaan Bhatia,Child Protection Law,Consumer Law; Immigration Law; White Collar Crime,primary,2500,3.6,8,New Delhi,Available for consultation,0.730889
4791,Adv. Akanksha Patel,Immigration & Citizenship Law,Competition Law; Criminal Law; Child Protection Law,secondary,2500,4.6,21,New Delhi,Limited availability,0.708889
1654,Adv. Priya Sharma,Labour & Employment Law,Criminal Law; Media and Entertainment Law; Child Protection Law,secondary,1500,4.9,2,New Delhi,Available for new matters,0.662444
3395,Adv. Dhruv Chopra,Criminal Law,Startup and Venture Capital; Securities Law; Child Protection Law,secondary,5000,4.3,14,New Delhi,Available for new matters,0.516000



User Prompt      : 'Car accident happened yesterday, looking for claim compensation lawyer'
User City (DB)   : Mumbai
Detected Domain  : ['Motor Accident Claims']
Total Matching Lawyers Found: 7  (primary: 2, secondary: 5)
ALL Matching Specialists (Primary + Secondary, Best & Most Affordable First):


,advocate_name,practice_area_primary,practice_area_secondary,match_tier,consultation_fee_inr,profile_rating,years_of_experience,city,availability,ranking_score
6961,Adv. Amit Bose,White Collar Crime,Civil Litigation; Securities Law; Motor Accident Claims,secondary,500,4.4,10,Mumbai,Limited availability,0.7460
4941,Adv. Sakshi Sen,Motor Accident Claims,Startup and Venture Capital; Trademark Law; Contract & Agreement Law,primary,1000,3.5,10,Mumbai,Available for new matters,0.7250
926,Adv. Rahul Sen,Data Protection & Privacy Law,Civil Litigation; Cyber Law; Motor Accident Claims,secondary,1000,4.5,33,Mumbai,Available for new matters,0.7150
1461,Adv. Priya Shah,Motor Accident Claims,Corporate Law; Insolvency and Bankruptcy; Contract & Agreement Law,primary,1500,3.7,39,Mumbai,Limited availability,0.6980
2387,Adv. Zoya Joshi,Copyright Law,Civil Litigation; Information Technology Law; Motor Accident Claims,secondary,1000,4.0,22,Mumbai,Available for consultation,0.6770
953,Adv. Priya Khanna,Cyber Law & IT,Civil Litigation; Constitutional Law; Motor Accident Claims,secondary,750,3.5,2,Mumbai,Limited availability,0.5995
262,Adv. Kavya Joshi,Contract & Agreement Law,Consumer Law; Patent Law; Motor Accident Claims,secondary,1000,4.2,6,Mumbai,Limited availability,0.5890



User Prompt      : 'Truck hit my bike on the highway, I need help claiming compensation'
User City (DB)   : 
Detected Domain  : ['Motor Accident Claims']
Total Matching Lawyers Found: 354  (primary: 68, secondary: 286)
Note: 0 primary specialists exist in this city for this domain -- every result
below was only surfaced because of the new secondary-match tier.


,advocate_name,practice_area_primary,practice_area_secondary,match_tier,consultation_fee_inr,profile_rating,years_of_experience,city,availability,ranking_score
1560,Adv. Trisha Iyer,Motor Accident Claims,Commercial Law; Immigration Law; Contract & Agreement Law,primary,500,4.2,37,Bengaluru,Available for consultation,0.968000
2784,Adv. Manish Reddy,Motor Accident Claims,Environmental Law; White Collar Crime; Contract & Agreement Law,primary,500,4.9,20,Chennai,Available for consultation,0.966000
3282,Adv. Aparna Bhatia,Motor Accident Claims,GST Law; Property Law; Contract & Agreement Law,primary,750,4.6,19,Dehradun,Available for consultation,0.939071
6846,Adv. Pallavi Chopra,Motor Accident Claims,Family Law; Real Estate Law; Contract & Agreement Law,primary,1500,4.2,27,Hyderabad,Available for new matters,0.932286
15,Adv. Vivek Malhotra,Motor Accident Claims,Media and Entertainment Law; Competition Law; Contract & Agreement Law,primary,750,4.9,14,Hyderabad,Available for consultation,0.921071
...,...,...,...,...,...,...,...,...,...,...
6931,Adv. Dev Kapoor,Cyber Law & IT,Civil Litigation; Media and Entertainment Law; Motor Accident Claims,secondary,7500,4.2,1,Chandigarh,Limited availability,0.434000
4965,Adv. Yash Sen,Banking & Finance Law,Civil Litigation; Real Estate Law; Motor Accident Claims,secondary,7500,3.9,1,Pune,Limited availability,0.422000
649,Adv. Rajat Bansal,Property & Land Law,Securities and Capital Markets; Civil Litigation; Motor Accident Claims,secondary,7500,3.6,2,Amaravati,Available for new matters,0.416000
3386,Adv. Tanvi Menon,Human Rights & PIL,Civil Litigation; Cyber Law; Motor Accident Claims,secondary,7500,3.6,2,Patna,Limited availability,0.416000


## 6. Evaluation Metrics
Evaluate Top-1 Accuracy, Top-3 Recall, and Mean Reciprocal Rank (MRR) on test dataset queries.

In [23]:
def evaluate_matching_performance(test_df):
    results = []
    for idx, row in test_df.iterrows():
        truth_domains = [d.strip() for d in str(row['legal_domain']).split('/')]
        pred_domains = predict_legal_domains(row['query_text'], top_k=3)
        
        top1_hit = pred_domains[0] in truth_domains
        top3_hit = any(d in truth_domains for d in pred_domains)
        
        mrr = 0.0
        for rank, d in enumerate(pred_domains, 1):
            if d in truth_domains:
                mrr = 1.0 / rank
                break
                
        results.append({'top1': int(top1_hit), 'top3': int(top3_hit), 'mrr': mrr})
        
    res_df = pd.DataFrame(results)
    print("=== SYSTEM EVALUATION METRICS ===")
    print(f"Top-1 Domain Accuracy : {res_df['top1'].mean()*100:.1f}%")
    print(f"Top-3 Domain Recall   : {res_df['top3'].mean()*100:.1f}%")
    print(f"Mean Reciprocal Rank  : {res_df['mrr'].mean():.3f}")

evaluate_matching_performance(test_queries)

=== SYSTEM EVALUATION METRICS ===
Top-1 Domain Accuracy : 38.1%
Top-3 Domain Recall   : 45.3%
Mean Reciprocal Rank  : 0.415


## 7. Coverage Impact: Primary-Only vs Primary+Secondary Matching
Quantifies how often the old primary-only engine would have had **zero relevant specialists** in a city for a given legal domain, and how many of those gaps are now filled by the secondary-match tier.


In [24]:
def coverage_impact_analysis():
    domains = adv['practice_area_primary'].unique()
    cities = adv['city'].unique()

    def sec_match(sec_val, domain):
        if pd.isna(sec_val):
            return False
        return domain in [p.strip() for p in str(sec_val).split(';')]

    total_pairs = 0
    zero_primary_pairs = 0
    rescued_by_secondary = 0
    gap_rows = []

    for city in cities:
        city_pool = adv[adv['city'].str.lower() == city.lower()]
        for domain in domains:
            total_pairs += 1
            primary_ct = (city_pool['practice_area_primary'] == domain).sum()
            if primary_ct == 0:
                zero_primary_pairs += 1
                secondary_ct = city_pool['practice_area_secondary'].apply(lambda s: sec_match(s, domain)).sum()
                if secondary_ct > 0:
                    rescued_by_secondary += 1
                    gap_rows.append({
                        'city': city,
                        'domain': domain,
                        'primary_matches': primary_ct,
                        'secondary_matches': int(secondary_ct)
                    })

    print("=== COVERAGE IMPACT: PRIMARY-ONLY vs PRIMARY+SECONDARY ===")
    print(f"Total (city x domain) combinations checked : {total_pairs}")
    print(f"Combinations with 0 primary specialists     : {zero_primary_pairs} "
          f"({zero_primary_pairs/total_pairs*100:.1f}%)")
    print(f"  -> of those, rescued by secondary matches : {rescued_by_secondary} "
          f"({rescued_by_secondary/max(zero_primary_pairs,1)*100:.1f}% of the gaps)")
    print(f"  -> still 0 matches even with secondary    : {zero_primary_pairs - rescued_by_secondary}")
    print()
    print("Sample of gaps filled by secondary matching:")
    return pd.DataFrame(gap_rows).sort_values('secondary_matches', ascending=False)

gap_report = coverage_impact_analysis()
display(gap_report.head(15))


=== COVERAGE IMPACT: PRIMARY-ONLY vs PRIMARY+SECONDARY ===
Total (city x domain) combinations checked : 1036
Combinations with 0 primary specialists     : 15 (1.4%)
  -> of those, rescued by secondary matches : 15 (100.0% of the gaps)
  -> still 0 matches even with secondary    : 0

Sample of gaps filled by secondary matching:


,city,domain,primary_matches,secondary_matches
8,New Delhi,Motor Accident Claims,0,23
9,Kolkata,Motor Accident Claims,0,23
0,Dehradun,Criminal Law,0,20
2,Jodhpur,Constitutional Law,0,19
12,Patna,Motor Accident Claims,0,16
11,Noida,Constitutional Law,0,16
4,Hyderabad,Child Protection Law,0,14
5,Gurugram,Child Protection Law,0,14
1,Jodhpur,Consumer Protection Law,0,13
13,Patna,Constitutional Law,0,11
